In [4]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

print("JAVA_HOME:", os.environ["JAVA_HOME"])

JAVA_HOME: C:\Program Files\Java\jdk-17


In [8]:
from pyspark.sql import SparkSession

In [9]:
spark = (
    SparkSession.builder
    .appName("CoffeeShopProject")
    .master("local[*]")
    .getOrCreate()
)
print("Spark version:", spark.version)
print("Spark session:", spark.sparkContext.appName)

Spark version: 4.2.0
Spark session: CoffeeShopProject


In [13]:
file_path = r"D:\Projects\Coffee Shop Sales - Cluster Computing\Dataset\Coffee Shop Sales.csv"
df = spark.read.csv(
    file_path,
    header = True,
    inferSchema = True
)

In [16]:
df.show(5)

+--------------+----------------+-------------------+---------------+--------+---------------+----------+----------+------------------+--------------------+--------------------+
|transaction_id|transaction_date|   transaction_time|transaction_qty|store_id| store_location|product_id|unit_price|  product_category|        product_type|      product_detail|
+--------------+----------------+-------------------+---------------+--------+---------------+----------+----------+------------------+--------------------+--------------------+
|             1|      01-01-2023|2026-10-20 07:06:11|              2|       5|Lower Manhattan|        32|       3.0|            Coffee|Gourmet brewed co...|         Ethiopia Rg|
|             2|      01-01-2023|2026-10-20 07:08:56|              2|       5|Lower Manhattan|        57|       3.1|               Tea|     Brewed Chai tea|Spicy Eye Opener ...|
|             3|      01-01-2023|2026-10-20 07:14:04|              2|       5|Lower Manhattan|        59|     

In [18]:
print("Total rows:", df.count())
print("Total columns:", len(df.columns))

df.printSchema()

Total rows: 149116
Total columns: 11
root
 |-- transaction_id: integer (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- transaction_time: timestamp (nullable = true)
 |-- transaction_qty: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- store_location: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- product_detail: string (nullable = true)



In [20]:
df.show(10, truncate=False)

+--------------+----------------+-------------------+---------------+--------+---------------+----------+----------+------------------+---------------------+---------------------------+
|transaction_id|transaction_date|transaction_time   |transaction_qty|store_id|store_location |product_id|unit_price|product_category  |product_type         |product_detail             |
+--------------+----------------+-------------------+---------------+--------+---------------+----------+----------+------------------+---------------------+---------------------------+
|1             |01-01-2023      |2026-10-20 07:06:11|2              |5       |Lower Manhattan|32        |3.0       |Coffee            |Gourmet brewed coffee|Ethiopia Rg                |
|2             |01-01-2023      |2026-10-20 07:08:56|2              |5       |Lower Manhattan|57        |3.1       |Tea               |Brewed Chai tea      |Spicy Eye Opener Chai Lg   |
|3             |01-01-2023      |2026-10-20 07:14:04|2              |5

In [21]:
df.describe().show()

+-------+-----------------+----------------+------------------+------------------+---------------+-----------------+-----------------+----------------+----------------+--------------------+
|summary|   transaction_id|transaction_date|   transaction_qty|          store_id| store_location|       product_id|       unit_price|product_category|    product_type|      product_detail|
+-------+-----------------+----------------+------------------+------------------+---------------+-----------------+-----------------+----------------+----------------+--------------------+
|  count|           149116|          149116|            149116|            149116|         149116|           149116|           149116|          149116|          149116|              149116|
|   mean|74737.37187156308|            NULL| 1.438276241315486| 5.342062555326055|           NULL|47.91860699053086|3.382219413074453|            NULL|            NULL|                NULL|
| stddev|43153.60001591786|            NULL|0.5425

In [24]:
from pyspark.sql.functions import col, sum
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+--------------+----------------+----------------+---------------+--------+--------------+----------+----------+----------------+------------+--------------+
|transaction_id|transaction_date|transaction_time|transaction_qty|store_id|store_location|product_id|unit_price|product_category|product_type|product_detail|
+--------------+----------------+----------------+---------------+--------+--------------+----------+----------+----------------+------------+--------------+
|             0|               0|               0|              0|       0|             0|         0|         0|               0|           0|             0|
+--------------+----------------+----------------+---------------+--------+--------------+----------+----------+----------------+------------+--------------+



In [27]:
from pyspark.sql.functions import col, sum, avg, count

In [29]:
df = df.withColumn(
    "revenue",
    col("transaction_qty") * col("unit_price")
)

df.select(
    "transaction_id",
    "product_detail",
    "transaction_qty",
    "unit_price",
    "revenue"
).show(10, truncate=False)

+--------------+---------------------------+---------------+----------+-------+
|transaction_id|product_detail             |transaction_qty|unit_price|revenue|
+--------------+---------------------------+---------------+----------+-------+
|1             |Ethiopia Rg                |2              |3.0       |6.0    |
|2             |Spicy Eye Opener Chai Lg   |2              |3.1       |6.2    |
|3             |Dark chocolate Lg          |2              |4.5       |9.0    |
|4             |Our Old Time Diner Blend Sm|1              |2.0       |2.0    |
|5             |Spicy Eye Opener Chai Lg   |2              |3.1       |6.2    |
|6             |Oatmeal Scone              |1              |3.0       |3.0    |
|7             |Our Old Time Diner Blend Sm|1              |2.0       |2.0    |
|8             |Columbian Medium Roast Sm  |2              |2.0       |4.0    |
|9             |Latte Rg                   |1              |4.25      |4.25   |
|10            |Dark chocolate Rg       

In [30]:
category_sales = (
    df.groupBy("product_category")
      .agg(
          sum("revenue").alias("total_revenue"),
          sum("transaction_qty").alias("total_quantity"),
          count("*").alias("transaction_rows")
      )
      .orderBy(col("total_revenue").desc())
)

category_sales.show(truncate=False)

+------------------+------------------+--------------+----------------+
|product_category  |total_revenue     |total_quantity|transaction_rows|
+------------------+------------------+--------------+----------------+
|Coffee            |269952.44999999146|89250         |58416           |
|Tea               |196405.9499999975 |69737         |45449           |
|Bakery            |82315.64000000006 |23214         |22796           |
|Drinking Chocolate|72416.0           |17457         |11468           |
|Coffee beans      |40085.25000000002 |1828          |1753            |
|Branded           |13607.0           |776           |747             |
|Loose Tea         |11213.59999999996 |1210          |1210            |
|Flavours          |8408.79999999989  |10511         |6790            |
|Packaged Chocolate|4407.639999999999 |487           |487             |
+------------------+------------------+--------------+----------------+



In [32]:
store_sales = (
    df.groupBy("store_location")
      .agg(
          sum("revenue").alias("total_revenue"),
          sum("transaction_qty").alias("total_quantity"),
          count("*").alias("transaction_rows")
      )
      .orderBy(col("total_revenue").desc())
)

store_sales.show(truncate=False)

+---------------+------------------+--------------+----------------+
|store_location |total_revenue     |total_quantity|transaction_rows|
+---------------+------------------+--------------+----------------+
|Hell's Kitchen |236511.1699999987 |71737         |50735           |
|Astoria        |232243.909999998  |70991         |50599           |
|Lower Manhattan|230057.24999999837|71742         |47782           |
+---------------+------------------+--------------+----------------+



In [33]:
from pyspark.sql.functions import (
    col, to_date, hour, dayofweek, month, dayofmonth
)

df = df.withColumn(
    "transaction_date",
    to_date(col("transaction_date"), "dd-MM-yyyy")
)

df.select("transaction_date", "transaction_time").show(5)

+----------------+-------------------+
|transaction_date|   transaction_time|
+----------------+-------------------+
|      2023-01-01|2026-10-20 07:06:11|
|      2023-01-01|2026-10-20 07:08:56|
|      2023-01-01|2026-10-20 07:14:04|
|      2023-01-01|2026-10-20 07:20:24|
|      2023-01-01|2026-10-20 07:22:41|
+----------------+-------------------+
only showing top 5 rows


In [34]:
df = (
    df.withColumn("hour", hour(col("transaction_time")))
      .withColumn("day_of_week", dayofweek(col("transaction_date")))
      .withColumn("month", month(col("transaction_date")))
      .withColumn("day_of_month", dayofmonth(col("transaction_date")))
)

df.select(
    "transaction_date",
    "hour",
    "day_of_week",
    "month",
    "day_of_month",
    "revenue"
).show(10)

+----------------+----+-----------+-----+------------+-------+
|transaction_date|hour|day_of_week|month|day_of_month|revenue|
+----------------+----+-----------+-----+------------+-------+
|      2023-01-01|   7|          1|    1|           1|    6.0|
|      2023-01-01|   7|          1|    1|           1|    6.2|
|      2023-01-01|   7|          1|    1|           1|    9.0|
|      2023-01-01|   7|          1|    1|           1|    2.0|
|      2023-01-01|   7|          1|    1|           1|    6.2|
|      2023-01-01|   7|          1|    1|           1|    3.0|
|      2023-01-01|   7|          1|    1|           1|    2.0|
|      2023-01-01|   7|          1|    1|           1|    4.0|
|      2023-01-01|   7|          1|    1|           1|   4.25|
|      2023-01-01|   7|          1|    1|           1|    7.0|
+----------------+----+-----------+-----+------------+-------+
only showing top 10 rows


In [35]:
from pyspark.sql.functions import sum, count

daily_sales = (
    df.groupBy("transaction_date", "store_id", "store_location")
      .agg(
          sum("revenue").alias("daily_revenue"),
          count("*").alias("transaction_count")
      )
      .orderBy("transaction_date", "store_id")
)

daily_sales.show(10)

+----------------+--------+---------------+-----------------+-----------------+
|transaction_date|store_id| store_location|    daily_revenue|transaction_count|
+----------------+--------+---------------+-----------------+-----------------+
|      2023-01-01|       3|        Astoria|            868.4|              190|
|      2023-01-01|       5|Lower Manhattan|788.3499999999999|              176|
|      2023-01-01|       8| Hell's Kitchen|851.4500000000002|              184|
|      2023-01-02|       3|        Astoria|925.5000000000001|              212|
|      2023-01-02|       5|Lower Manhattan|649.0500000000001|              159|
|      2023-01-02|       8| Hell's Kitchen|828.8000000000001|              195|
|      2023-01-03|       3|        Astoria|902.7500000000003|              205|
|      2023-01-03|       5|Lower Manhattan|            756.0|              166|
|      2023-01-03|       8| Hell's Kitchen|           906.25|              211|
|      2023-01-04|       3|        Astor

In [36]:
print("Daily sales rows:", daily_sales.count())

daily_sales.printSchema()

Daily sales rows: 543
root
 |-- transaction_date: date (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- store_location: string (nullable = true)
 |-- daily_revenue: double (nullable = true)
 |-- transaction_count: long (nullable = false)



In [38]:
from pyspark.sql.functions import dayofweek, month, dayofmonth

daily_sales = (
    daily_sales
    .withColumn("day_of_week", dayofweek("transaction_date"))
    .withColumn("month", month("transaction_date"))
    .withColumn("day_of_month", dayofmonth("transaction_date"))
)

daily_sales.select(
    "transaction_date",
    "store_id",
    "day_of_week",
    "month",
    "day_of_month",
    "daily_revenue"
).show(10)

+----------------+--------+-----------+-----+------------+-----------------+
|transaction_date|store_id|day_of_week|month|day_of_month|    daily_revenue|
+----------------+--------+-----------+-----+------------+-----------------+
|      2023-01-01|       3|          1|    1|           1|            868.4|
|      2023-01-01|       5|          1|    1|           1|788.3499999999999|
|      2023-01-01|       8|          1|    1|           1|851.4500000000002|
|      2023-01-02|       3|          2|    1|           2|925.5000000000001|
|      2023-01-02|       5|          2|    1|           2|649.0500000000001|
|      2023-01-02|       8|          2|    1|           2|828.8000000000001|
|      2023-01-03|       3|          3|    1|           3|902.7500000000003|
|      2023-01-03|       5|          3|    1|           3|            756.0|
|      2023-01-03|       8|          3|    1|           3|           906.25|
|      2023-01-04|       3|          4|    1|           4|808.2500000000003|

In [39]:
from pyspark.sql.functions import col, lit

train_data = daily_sales.filter(
    col("transaction_date") < lit("2023-06-01")
)

test_data = daily_sales.filter(
    col("transaction_date") >= lit("2023-06-01")
)

print("Training rows:", train_data.count())
print("Testing rows:", test_data.count())

Training rows: 453
Testing rows: 90
